# Inventory QuoteAI Unity Catalog Objects

Read-only inventory of schemas matching `dev_quote_ai*` in the `deepwater_tst` catalog. It includes schemas, tables and views (including materialized or streaming tables), volumes, and routines (functions/procedures) visible to the identity running this notebook.

Unity Catalog metadata only shows objects the active principal is permitted to see. This notebook does not create, alter, or delete catalog objects. CSV export is optional and requires a writable Unity Catalog Volume path.

## 1. Parameters and Dependencies

The notebook uses the Databricks-provided Spark session and current workspace identity; no database password, token, or external connector is needed. Set `export_volume_path` to a writable UC Volume directory to save a CSV; leave it blank to only display the inventory.

In [ ]:
import re
from datetime import datetime, timezone

from pyspark.sql import functions as F

DEFAULT_CATALOG = "deepwater_tst"
DEFAULT_SCHEMA_PREFIX = "dev_quote_ai"
IDENTIFIER_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")

# Remove old widgets so the notebook can be rerun with changed parameters.
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", DEFAULT_CATALOG, "Catalog")
dbutils.widgets.text("schema_prefix", DEFAULT_SCHEMA_PREFIX, "Schema name prefix")
dbutils.widgets.text("export_volume_path", "", "Optional writable UC Volume directory for CSV export")

catalog_name = dbutils.widgets.get("catalog").strip()
schema_prefix = dbutils.widgets.get("schema_prefix").strip()
export_volume_path = dbutils.widgets.get("export_volume_path").strip()

if not IDENTIFIER_PATTERN.fullmatch(catalog_name):
    raise ValueError("Catalog must be a simple Unity Catalog identifier")
if not IDENTIFIER_PATTERN.fullmatch(schema_prefix):
    raise ValueError("Schema prefix must be a simple identifier prefix")

quoted_catalog = f"`{catalog_name}`"

## 2. Validate the Active Databricks Connection

This reports the catalog, schema, and principal for the current Spark session. The notebook does not open a second connection or request credentials.

In [ ]:
connection_info = spark.sql(
    "SELECT current_catalog() AS current_catalog, "
    "current_schema() AS current_schema, current_user() AS current_user"
)
display(connection_info)

## 3. Find Matching Schemas

Unity Catalog `INFORMATION_SCHEMA.SCHEMATA` is scoped to the selected catalog. Matching is a literal prefix comparison in Python, avoiding SQL wildcard ambiguity.

In [ ]:
schema_metadata = spark.sql(
    f"""
    SELECT catalog_name, schema_name, schema_owner, comment, created, last_altered
    FROM {quoted_catalog}.information_schema.schemata
    WHERE catalog_name = '{catalog_name}'
      AND substr(schema_name, 1, length('{schema_prefix}')) = '{schema_prefix}'
    ORDER BY schema_name
    """
)
matching_schemas = [
    row.asDict(recursive=True)
    for row in schema_metadata.collect()
    if row["schema_name"].startswith(schema_prefix)
]
schema_names = [row["schema_name"] for row in matching_schemas]

print(f"Found {len(schema_names)} visible schemas matching {catalog_name}.{schema_prefix}*")
display(schema_metadata.filter(F.col("schema_name").startswith(schema_prefix)))

## 4. Ingestion Method Reference

Maps each known QuoteAI object to how its data is retrieved, taken from `config/quote_ai/ingest/sources.yml` and `src/quote_ai/ingest`. Schemas are matched by logical name (a `dev_` or `dev_<user>_` prefix is ignored). Objects with no entry here are not loaded by the ingestion framework. Update this list when `sources.yml` changes.

In [ ]:
GUDID_FULL = (
    "File download: HTTPS GET of the AccessGUDID monthly delimited full release zip into the quote_ai_releases "
    "volume, then Auto Loader (pipe-delimited CSV) in the bronze pipeline with a keyed AUTO CDC upsert; source "
    "gudid_full_release, file {file}"
)
SYNTHETIC = (
    "Synthetic data: generated by quote_ai.ingest.synthetic (source dbxlab_synthetic) as CSV into the "
    "quote_ai_landing volume, then Auto Loader (CSV) in the bronze pipeline; no external API"
)
OPS_WRITTEN = "Written by the ingestion code with Spark SQL (no external source): {what}"

# (logical schema, object name, how the data is retrieved)
INGESTION_METHODS = [
    ("quote_ai_gudid_bronze", "device_raw", GUDID_FULL.format(file="device.txt")),
    ("quote_ai_gudid_bronze", "identifiers_raw", GUDID_FULL.format(file="identifiers*.txt")),
    ("quote_ai_gudid_bronze", "device_sizes_raw", GUDID_FULL.format(file="deviceSizes*.txt")),
    ("quote_ai_gudid_bronze", "product_codes_raw", GUDID_FULL.format(file="productCodes*.txt")),
    ("quote_ai_gudid_bronze", "gmdn_terms_raw", GUDID_FULL.format(file="gmdnTerms*.txt")),
    (
        "quote_ai_gudid_bronze",
        "daily_device_raw",
        "File download: HTTPS GET of the AccessGUDID daily update zip (gudid_daily_update_YYYYMMDD.zip) into the "
        "quote_ai_releases volume, then Auto Loader (XML, rowTag device) with a keyed AUTO CDC upsert; "
        "source gudid_daily_updates",
    ),
    (
        "quote_ai_gudid_bronze",
        "api_device_history",
        "REST API: AccessGUDID v3 GET /api/v3/devices/history.json?record_key=... via the quote_ai_gudid_api "
        "PySpark data source (max 2 requests/s, retries with backoff, at most 2000 requests per run), for "
        "glove-category keys from daily_device_raw; merged by record key and version; source gudid_api_device_history",
    ),
    ("quote_ai_sales_bronze", "price_book_raw", SYNTHETIC),
    ("quote_ai_sales_bronze", "inventory_raw", SYNTHETIC),
    ("quote_ai_sales_bronze", "customer_raw", SYNTHETIC),
    (
        "quote_ai_gudid_bronze",
        "quote_ai_releases",
        "Landing zone: files written by the HTTPS download step (zips as published plus extracted files)",
    ),
    (
        "quote_ai_sales_bronze",
        "quote_ai_landing",
        "Landing zone: synthetic CSV files written by the dbxlab_synthetic generator",
    ),
    ("quote_ai_ops", "ingestion_log", OPS_WRITTEN.format(what="one row per source run")),
    ("quote_ai_ops", "ingestion_file", OPS_WRITTEN.format(what="one row per landed file, with size, MD5 and SHA-256")),
    ("quote_ai_ops", "ingestion_config", OPS_WRITTEN.format(what="the sources.yml config in force, with history")),
    ("quote_ai_ops", "v_ingestion_audit", "View over ingestion_log, ingestion_file and ingestion_config"),
    (
        "quote_ai_ops",
        "glove_classification",
        "Derived: classification rules from sources.yml applied to the GUDID bronze tables; "
        "no API call at run time",
    ),
    ("quote_ai_ops", "glove_devices", "View over glove_classification, included devices only"),
]
SCHEMA_METHOD = "Created by the bundle (resources/quote_ai/*.yml); holds objects, no data is ingested into it directly"
UNMAPPED_METHOD = "Not defined in sources.yml: not loaded by the ingestion framework"


def ingestion_method_column():
    """Column expression giving each inventory row its ingestion method."""
    expression = F.when(F.col("object_type") == "SCHEMA", F.lit(SCHEMA_METHOD))
    for schema, name, method in INGESTION_METHODS:
        matches = (F.col("object_name") == name) & F.col("schema_name").endswith(schema)
        expression = expression.when(matches, F.lit(method))
    return expression.otherwise(F.lit(UNMAPPED_METHOD))

## 5. Inventory Objects in Matching Schemas

The inventory includes schema metadata, relations from `TABLES` (tables, views, materialized views, streaming tables), Unity Catalog volumes, and routines (functions/procedures). The `ingestion_method` column says which API or file download loads each object, or how it is derived. Metadata visibility follows the current user’s Unity Catalog privileges.

In [ ]:
schema_filter = ", ".join("'" + name.replace("'", "''") + "'" for name in schema_names)
objects = schema_metadata.filter(F.col("schema_name").startswith(schema_prefix)).select(
    F.col("catalog_name").alias("catalog_name"),
    F.col("schema_name").alias("schema_name"),
    F.col("schema_name").alias("object_name"),
    F.lit("SCHEMA").alias("object_type"),
    F.concat_ws(
        " | ",
        F.concat(F.lit("owner="), F.col("schema_owner")),
        F.concat(F.lit("comment="), F.coalesce(F.col("comment"), F.lit(""))),
    ).alias("details"),
    F.col("created").alias("created"),
    F.col("last_altered").alias("last_altered"),
)

if schema_names:
    tables = spark.sql(
        f"""
        SELECT table_catalog AS catalog_name, table_schema AS schema_name, table_name AS object_name,
               table_type AS object_type,
               concat_ws(' | ', table_type, coalesce(comment, '')) AS details,
               created, last_altered
        FROM {quoted_catalog}.information_schema.tables
        WHERE table_schema IN ({schema_filter})
        """
    )
    volumes = spark.sql(
        f"""
        SELECT volume_catalog AS catalog_name, volume_schema AS schema_name, volume_name AS object_name,
               'VOLUME' AS object_type,
               concat_ws(' | ', volume_type, concat('owner=', volume_owner), coalesce(comment, '')) AS details,
               created, last_altered
        FROM {quoted_catalog}.information_schema.volumes
        WHERE volume_schema IN ({schema_filter})
        """
    )
    routines = spark.sql(
        f"""
        SELECT routine_catalog AS catalog_name, routine_schema AS schema_name, routine_name AS object_name,
               routine_type AS object_type,
               concat_ws(' | ', coalesce(data_type, ''), coalesce(comment, '')) AS details,
               created, last_altered
        FROM {quoted_catalog}.information_schema.routines
        WHERE routine_schema IN ({schema_filter})
        """
    )
    objects = objects.unionByName(tables).unionByName(volumes).unionByName(routines)

object_inventory = objects.withColumn("ingestion_method", ingestion_method_column()).orderBy(
    "schema_name", "object_type", "object_name"
)
print(f"Inventory contains {object_inventory.count()} visible objects across {len(schema_names)} matching schemas.")
display(object_inventory)

## 6. Summary by Schema and Object Type

In [ ]:
object_counts = (
    object_inventory.groupBy("schema_name", "object_type")
    .count()
    .orderBy("schema_name", "object_type")
)
display(object_counts)

## 7. Optional CSV Export

Set `export_volume_path` to a writable UC Volume path such as `/Volumes/deepwater_tst/<schema>/<volume>/inventory`. The notebook writes a timestamped CSV directory and prints the exact CSV part-file path. If left blank, the displayed tables remain available in the notebook and no file is written.

In [ ]:
from pathlib import PurePosixPath

if not export_volume_path:
    print("CSV export skipped. Set export_volume_path to a writable /Volumes/<catalog>/<schema>/<volume> path.")
else:
    volume_path = PurePosixPath(export_volume_path)
    parts = volume_path.parts
    if len(parts) < 5 or parts[0] != "/" or parts[1] != "Volumes" or parts[2] != catalog_name:
        raise ValueError(f"Export path must be under /Volumes/{catalog_name}/<schema>/<volume>")
    if not IDENTIFIER_PATTERN.fullmatch(parts[3]) or not IDENTIFIER_PATTERN.fullmatch(parts[4]):
        raise ValueError("Export path must contain valid schema and volume identifiers")
    if any(part in (".", "..") for part in parts[5:]):
        raise ValueError("Export path cannot contain dot or parent-directory segments")

    export_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    export_directory = str(volume_path / f"dev_quote_ai_object_inventory_{export_timestamp}")
    (object_inventory.coalesce(1)
     .write.mode("errorifexists")
     .option("header", "true")
     .csv(export_directory))
    csv_files = [entry.path for entry in dbutils.fs.ls(export_directory) if entry.name.endswith(".csv")]
    if len(csv_files) != 1:
        raise RuntimeError(f"Expected one CSV part file in {export_directory}; found {len(csv_files)}")
    print(f"CSV inventory saved to: {csv_files[0]}")